# Experiment 2: Which vectorization?

Bag of Words vs TF-IDF, each with unigrams (1,1), bigrams (1,2) and trigrams (1,3).
5,000 features and the same Random Forest for every run.

In [1]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000'))
EXPERIMENT = 'Exp 2 - BoW vs TfIdf'
mlflow.set_experiment(EXPERIMENT)
BATCH = datetime.now().strftime('%Y%m%d-%H%M%S')  # tags this notebook run's MLflow runs

df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)
df.shape

2026/09/11 11:03:08 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf' does not exist. Creating a new experiment.


(36662, 2)

In [2]:
def log_evaluation(y_true, y_pred, title):
    """Log accuracy, per-class metrics and a confusion matrix to the active MLflow run."""
    mlflow.log_metric('accuracy', accuracy_score(y_true, y_pred))
    for label, metrics in classification_report(y_true, y_pred, output_dict=True).items():
        if isinstance(metrics, dict):
            mlflow.log_metrics({f'{label}_{name}': value for name, value in metrics.items()})

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set(xlabel='Predicted', ylabel='Actual', title=f'Confusion Matrix: {title}')
    mlflow.log_figure(fig, 'confusion_matrix.png')
    plt.close(fig)


def compare_runs():
    """Table of this batch's runs; the video reads the same numbers off MLflow's parallel-coordinates plot."""
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.batch = '{BATCH}'")
    columns = {
        'tags.mlflow.runName': 'run',
        'metrics.accuracy': 'accuracy',
        'metrics.-1_precision': 'neg_precision',
        'metrics.-1_recall': 'neg_recall',
        'metrics.1_precision': 'pos_precision',
        'metrics.1_recall': 'pos_recall',
    }
    return runs[list(columns)].rename(columns=columns).sort_values('accuracy', ascending=False).round(4)

In [3]:
def run_experiment(vectorizer_name, ngram_range, max_features):
    vectorizer_cls = CountVectorizer if vectorizer_name == 'BoW' else TfidfVectorizer
    vectorizer = vectorizer_cls(ngram_range=ngram_range, max_features=max_features)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    with mlflow.start_run(run_name=f'{vectorizer_name}_{ngram_range}_RandomForest'):
        mlflow.set_tags({'experiment_type': 'feature_engineering', 'model_type': 'RandomForestClassifier', 'batch': BATCH})
        mlflow.log_params({
            'vectorizer_type': vectorizer_name,
            'ngram_range': ngram_range,
            'vectorizer_max_features': max_features,
            'n_estimators': 200,
            'max_depth': 15,
        })

        model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        log_evaluation(y_test, model.predict(X_test), f'{vectorizer_name}, {ngram_range}')


for ngram_range in [(1, 1), (1, 2), (1, 3)]:
    for vectorizer_name in ['BoW', 'TF-IDF']:
        run_experiment(vectorizer_name, ngram_range, max_features=5000)

2026/09/11 11:03:10 INFO mlflow.tracking._tracking_service.client: 🏃 View run BoW_(1, 1)_RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/ec53ed1f3a5e4f74883e905c6f5121d3.


2026/09/11 11:03:10 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2.


2026/09/11 11:03:13 INFO mlflow.tracking._tracking_service.client: 🏃 View run TF-IDF_(1, 1)_RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/0dc4feb46973400fa0bab49b6403d623.


2026/09/11 11:03:13 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2.


2026/09/11 11:03:18 INFO mlflow.tracking._tracking_service.client: 🏃 View run BoW_(1, 2)_RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/ed501cd9a24e4284922252861e2738d6.


...

2026/09/11 11:03:22 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2.


2026/09/11 11:03:29 INFO mlflow.tracking._tracking_service.client: 🏃 View run BoW_(1, 3)_RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/7c7aa10aa5cd4a7eae5a274e94cabc1c.


2026/09/11 11:03:29 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2.


2026/09/11 11:03:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run TF-IDF_(1, 3)_RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/fcf376e061234629abebb397d496dae7.


2026/09/11 11:03:36 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2.


In [4]:
compare_runs()

,run,accuracy,neg_precision,neg_recall,pos_precision,pos_recall
2,"TF-IDF_(1, 2)_RandomForest",0.6529,0.9216,0.0285,0.6366,0.8354
3,"BoW_(1, 2)_RandomForest",0.6486,0.9737,0.0224,0.6339,0.8250
0,"TF-IDF_(1, 3)_RandomForest",0.6473,0.9714,0.0206,0.6375,0.8209
1,"BoW_(1, 3)_RandomForest",0.6469,0.9643,0.0164,0.6364,0.8253
5,"BoW_(1, 1)_RandomForest",0.6460,0.9583,0.0139,0.6355,0.8231
4,"TF-IDF_(1, 1)_RandomForest",0.6443,0.9583,0.0139,0.6346,0.8253


The video picks **TF-IDF with trigrams (1,3)** because it gives the best negative-class recall.
Accuracy alone would favour a model that rarely predicts the minority class.